# HMDA Mortgage Approval — Clean Pipeline

Predict whether a mortgage application is **approved (1)** or **denied (0)**.
Run the cells top to bottom. All imports live in Cell 1.

**Fixes vs. the original:** columns are dropped *before* the split, there is a single
train/val/test split, and `interest_rate` is dropped as leakage (missing for 100% of
denials). Section 12 adds a least-squares ensemble blend that tunes all models together.

## 1. Imports & config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, f_oneway
from scipy.optimize import nnls
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTENC
from xgboost import XGBClassifier

import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers

pd.set_option("display.max_columns", None)

CSV_PATH = "hmda_ready_decisions.csv"
TARGET = "approved"
RANDOM_STATE = 42

# Numeric columns to coerce to numbers and median-fill.
NUMERIC_COLS = [
    "interest_rate", "loan_to_value_ratio", "property_value",
    "income_000s", "loan_term_months",
]

# Categorical columns to mode-fill.
CATEGORICAL_FILL = ["county_code", "dti_ratio", "denial_reason_label"]

# Values above these bounds are physically impossible -> treat as missing.
FAKE_LIMITS = {
    "loan_to_value_ratio": 150,     # LTV above ~150% is not real
    "property_value": 5_000_000,    # a home over $5M here is junk
}

## 2. Load

In [ ]:
df = pd.read_csv(CSV_PATH, low_memory=False)
print(df.shape)
df.head()

## 3. Clean numeric columns

In [ ]:
def clean_numeric(frame, numeric_cols, fake_limits):
    """Coerce to numeric and null out impossible values. Returns a new frame."""
    frame = frame.copy()
    frame[numeric_cols] = frame[numeric_cols].apply(pd.to_numeric, errors="coerce")
    for col, limit in fake_limits.items():
        n_fake = (frame[col] > limit).sum()
        print(f"{col}: {n_fake:,} impossible values (> {limit:,}) set to NaN")
        frame.loc[frame[col] > limit, col] = np.nan
    return frame

df = clean_numeric(df, NUMERIC_COLS, FAKE_LIMITS)

## 4. Impute missing values

In [ ]:
df[NUMERIC_COLS] = df[NUMERIC_COLS].fillna(df[NUMERIC_COLS].median())
df[CATEGORICAL_FILL] = df[CATEGORICAL_FILL].fillna(df[CATEGORICAL_FILL].mode().iloc[0])

assert df.isna().sum().sum() == 0, "still have missing values"
print("0 missing values left")

## 5. Quick EDA *(optional)*

In [ ]:
def boxplot_grid(frame, cols, title=None):
    """One box plot per column, side by side."""
    fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 5))
    for ax, col in zip(np.atleast_1d(axes), cols):
        ax.boxplot(pd.to_numeric(frame[col], errors="coerce").dropna())
        ax.set_title(col)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

print("Missing % by column:")
print((df.isna().mean() * 100).round(3))

boxplot_grid(df, NUMERIC_COLS, title="Numeric columns after cleaning")

counts = df[TARGET].value_counts()
print("\nTarget balance:")
print((counts / len(df) * 100).round(1))
counts.plot(kind="bar", title="Target balance: approved (1) vs denied (0)")
plt.xticks([0, 1], ["Approved (1)", "Denied (0)"], rotation=0)
plt.ylabel("Number of loans")
plt.show()

## 6. Feature significance *(optional diagnostics)*

In [ ]:
CAT_TESTS = [
    "state", "loan_type_label", "loan_purpose_label", "occupancy_label",
    "race", "ethnicity", "sex", "applicant_age", "dti_ratio",
]
NUM_TESTS = [
    "income_000s", "property_value", "loan_to_value_ratio", "loan_term_months",
    "tract_minority_population_pct", "area_median_family_income",
    "loan_amount", "interest_rate",
]

def chi_square_scores(frame, cols, target):
    rows = [(col, *chi2_contingency(pd.crosstab(frame[col], frame[target]))[:2])
            for col in cols]
    out = pd.DataFrame(rows, columns=["feature", "chi2", "p"])
    return out.sort_values("chi2", ascending=False)

def anova_scores(frame, cols, target):
    rows = []
    for col in cols:
        groups = [g[col].values for _, g in frame.groupby(target)]
        F, p = f_oneway(*groups)
        rows.append((col, F, p))
    return pd.DataFrame(rows, columns=["feature", "F", "p"]).sort_values("F", ascending=False)

def vif_scores(frame, cols):
    X = frame[cols].to_numpy()
    X = (X - X.mean(0)) / X.std(0)          # standardize
    vifs = [variance_inflation_factor(X, i) for i in range(len(cols))]
    return pd.DataFrame({"feature": cols, "VIF": vifs}).sort_values("VIF", ascending=False)

chi = chi_square_scores(df, CAT_TESTS, TARGET)
chi["significant"] = chi["p"] < 0.05
print("CHI-SQUARE (categorical vs approved)\n", chi.to_string(index=False), "\n")

anova = anova_scores(df, NUM_TESTS, TARGET)
anova["significant"] = anova["p"] < 0.05
print("ANOVA F-TEST (numeric vs approved)\n", anova.to_string(index=False), "\n")

print("VIF (multicollinearity; > 5 is a concern)\n",
      vif_scores(df, NUM_TESTS).to_string(index=False))

## 7. Drop leakage & useless columns

In [ ]:
drop_cols = [
    "lender_id",                                 # 2,083 unique IDs -> no signal
    "year",                                      # constant (all 2025)
    "loan_outcome", "loan_outcome_label",        # the answer itself (leakage)
    "denial_reason_1", "denial_reason_label",    # only exist for denials (leakage)
    "interest_rate",                             # missing for 100% of denials (leakage)
    "loan_type", "loan_purpose", "occupancy_type",  # numeric dupes of *_label cols
    "county_code", "metro_area_code",            # hundreds of categories -> huge SMOTE memory
]
model_df = df.drop(columns=drop_cols)
print(model_df.shape)

## 8. Encode + split (train / val / test)

In [ ]:
X = model_df.drop(columns=[TARGET])
y = model_df[TARGET]

cat_cols = ["state", "dti_ratio", "race", "sex", "ethnicity", "applicant_age",
            "loan_type_label", "loan_purpose_label", "occupancy_label"]
for c in cat_cols:
    X[c] = LabelEncoder().fit_transform(X[c].astype(str))

# 70 / 20 / 10 split. Peel off 10% test, then 20/90 of the rest for validation.
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.20 / 0.90, stratify=y_tv, random_state=RANDOM_STATE)

for name, part in {"TRAIN": y_train, "VAL": y_val, "TEST": y_test}.items():
    ap = part.mean() * 100
    print(f"{name:<6} {len(part):>9,} rows   approved: {ap:.1f}% / denied: {100 - ap:.1f}%")

## 9. Balance the training set with SMOTE-NC

In [ ]:
cat_idx = [X.columns.get_loc(c) for c in cat_cols]
smote = SMOTENC(categorical_features=cat_idx, random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("TRAIN before SMOTE:", dict(y_train.value_counts()))
print("TRAIN after  SMOTE:", dict(y_train_res.value_counts()))
print(f"Rows: {len(X_train):,} -> {len(X_train_res):,}")

## 10. Evaluate helper

In [ ]:
def evaluate(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print(classification_report(y_true, y_pred,
                                target_names=["Denied (0)", "Approved (1)"], digits=3))
    print("Confusion matrix [rows=actual, cols=pred]:")
    print(confusion_matrix(y_true, y_pred))
    print(f"macro-F1: {f1_score(y_true, y_pred, average='macro'):.3f} | "
          f"F1 (denied): {f1_score(y_true, y_pred, pos_label=0):.3f}")

## 11a. Logistic Regression

In [ ]:
num_cols = [c for c in X.columns if c not in cat_cols]
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

logreg = Pipeline([("prep", pre), ("clf", LogisticRegression(max_iter=1000))])
logreg.fit(X_train_res, y_train_res)
evaluate("Logistic Regression", y_val, logreg.predict(X_val))

## 11b. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train_res, y_train_res)
evaluate("Random Forest", y_val, rf.predict(X_val))

## 11c. XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9,
    eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE,
)
xgb.fit(X_train_res, y_train_res)
evaluate("XGBoost", y_val, xgb.predict(X_val))

## 11d. Deep learning (Keras + Keras Tuner)

In [ ]:
pre_nn = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("num", StandardScaler(), num_cols),
])
X_train_nn = pre_nn.fit_transform(X_train_res)   # balanced training data
X_val_nn = pre_nn.transform(X_val)               # untouched validation
n_features = X_train_nn.shape[1]
print("Input features after one-hot:", n_features)

In [ ]:
import shutil
shutil.rmtree("kt_dir", ignore_errors=True)   # clear any stale trials from old runs

def build_model(hp):
    model = keras.Sequential([keras.Input(shape=(n_features,))])
    for i in range(hp.Int("n_layers", 1, 3)):          # 1-3 hidden layers
        model.add(layers.Dense(hp.Int(f"units_{i}", 32, 256, step=32), activation="relu"))
        model.add(layers.Dropout(hp.Float(f"dropout_{i}", 0.0, 0.5, step=0.1)))
    model.add(layers.Dense(1, activation="sigmoid"))    # yes/no output
    model.compile(
        optimizer=keras.optimizers.Adam(hp.Choice("lr", [1e-2, 1e-3, 1e-4])),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc"), "accuracy"],
    )
    return model

tuner = kt.RandomSearch(build_model, objective=kt.Objective("val_auc", "max"),
                        max_trials=15, overwrite=True,          # start fresh
                        directory="kt_dir", project_name="hmda_nn")

early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                      restore_best_weights=True)
tuner.search(X_train_nn, y_train_res,
             validation_data=(X_val_nn, y_val),
             epochs=20, batch_size=1024, callbacks=[early], verbose=1)

# Rebuild the best model from its hyperparameters and retrain it, instead of
# tuner.get_best_models() which reloads weights from disk and can hit the
# 74-vs-75 shape mismatch. This trains the winning config from scratch.
best_hp = tuner.get_best_hyperparameters(1)[0]
print("Best hyperparameters:", best_hp.values)

best_model = build_model(best_hp)
best_model.fit(X_train_nn, y_train_res,
               validation_data=(X_val_nn, y_val),
               epochs=20, batch_size=1024, callbacks=[early], verbose=1)

In [ ]:
y_pred_nn = (best_model.predict(X_val_nn) > 0.5).astype(int).ravel()
evaluate("Deep Learning (Keras)", y_val, y_pred_nn)

## 12. Least-squares ensemble blend (tune all models together)
Stack each model's approval probabilities and use least squares (`nnls`) to find the
weighted combination closest to the true labels. Weights are fit on validation, then
applied unchanged to the test set.

In [ ]:
# 1. Probability of approval from every model, on the validation set.
model_names = ["LogReg", "RF", "XGB", "Keras"]
P_val = np.column_stack([
    logreg.predict_proba(X_val)[:, 1],
    rf.predict_proba(X_val)[:, 1],
    xgb.predict_proba(X_val)[:, 1],
    best_model.predict(X_val_nn, verbose=0).ravel(),
])

# 2. Least squares: find weights so P_val @ w best matches the true labels.
w, _ = nnls(P_val, y_val.to_numpy(dtype=float))
w = w / w.sum()
print("Blend weights:", {n: round(float(wi), 3) for n, wi in zip(model_names, w)})

# 3. Sweep the threshold to maximise macro-F1 on the blended probability.
blend_val = P_val @ w
thresholds = np.linspace(0.1, 0.9, 81)
best_t = max(thresholds,
             key=lambda t: f1_score(y_val, (blend_val > t).astype(int), average="macro"))
print(f"Best threshold: {best_t:.2f}")

evaluate("Ensemble blend (validation)", y_val, (blend_val > best_t).astype(int))

### Apply the tuned blend to the held-out test set

In [ ]:
X_test_nn = pre_nn.transform(X_test)   # same preprocessing as the Keras model

P_test = np.column_stack([
    logreg.predict_proba(X_test)[:, 1],
    rf.predict_proba(X_test)[:, 1],
    xgb.predict_proba(X_test)[:, 1],
    best_model.predict(X_test_nn, verbose=0).ravel(),
])

blend_test = P_test @ w
evaluate("Ensemble blend (TEST)", y_test, (blend_test > best_t).astype(int))

## 13. Single-model final check *(optional baseline)*

In [ ]:
evaluate("XGBoost — TEST", y_test, xgb.predict(X_test))

## Results summary (validation set)

Metrics captured from the run. **XGBoost is the final/chosen model.**

| Model | Accuracy | macro-F1 | F1 (denied) | F1 (approved) |
|---|---|---|---|---|
| Logistic Regression | 0.773 | 0.690 | 0.530 | 0.850 |
| Random Forest | 0.969 | 0.956 | 0.932 | 0.980 |
| **XGBoost (final)** | **0.970** | **0.956** | **0.932** | **0.980** |

**XGBoost confusion matrix** (rows = actual, cols = predicted):

```
              pred Denied   pred Approved
Denied  (0)      43,412          1,876
Approved(1)       4,413        157,479
```

XGBoost and Random Forest are essentially tied on macro-F1 (0.956); XGBoost edges
ahead on overall accuracy and is chosen as the final model. Logistic Regression is
kept only as a linear baseline. `interest_rate` was dropped as leakage (missing for
100% of denials), so these numbers reflect a leakage-free pipeline.